# 🚀 Stable‑Dreamfusion – يعمل 100% على Colab
**تثبيت ذكي، بناء موثوق، واجهة Gradio – كل شيء جاهز من أول خلية.**

## 1. استنساخ المستودع

In [ ]:
import os
if not os.path.exists('/content/stable-dreamfusion'):
    %cd /content
    !git clone https://github.com/ashawkey/stable-dreamfusion.git
    %cd /content/stable-dreamfusion
    !git checkout 4171f00c8d1721bb4645bad902b8b4d6fae3cef5
else:
    %cd /content/stable-dreamfusion
    print('✅ المستودع موجود مسبقًا، تخطي التحميل')

## 2. تجهيز البيئة – ضمان التوافق التام

In [ ]:
import subprocess, sys, warnings, os, urllib.request
warnings.filterwarnings('ignore')

# ---------- إصلاح مصادر apt ----------
if os.path.exists('/etc/apt/sources.list.d/r2u.list'):
    !sudo rm /etc/apt/sources.list.d/r2u.list
    !apt-get update -qq 2>/dev/null

# ---------- أدوات النظام ----------
!apt-get install -y ffmpeg libopencv-dev libglm-dev libxrender1 libgl1-mesa-glx libegl1-mesa libegl1-mesa-dev > /dev/null 2>&1

# ---------- إزالة PyTorch القديم (إذا كان مثبتًا مع CUDA غير مناسب) ----------
!pip uninstall -y torch torchvision torchaudio

# ---------- تنزيل وتثبيت CUDA toolkit 11.8 (إذا لم يكن موجودًا) ----------
def get_cuda_version():
    try:
        nvcc_out = subprocess.check_output(['nvcc', '--version']).decode()
        if 'release 11.8' in nvcc_out:
            return '11.8'
        else:
            return nvcc_out.split('release ')[1].split(',')[0].strip()
    except:
        return None

cuda_ver = get_cuda_version()
if cuda_ver != '11.8':
    print(f'⏳ تثبيت CUDA 11.8 (الحالي: {cuda_ver}) ...')
    !wget -q https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
    !sh cuda_11.8.0_520.61.05_linux.run --toolkit --silent --override
    !rm cuda_11.8.0_520.61.05_linux.run
else:
    print('✅ CUDA 11.8 موجود بالفعل')

# ضبط المتغيرات البيئية
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# التأكد من nvcc
!nvcc --version

# ---------- تثبيت PyTorch مع CUDA 11.8 ----------
!pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q setuptools==70.0.0

# ---------- دالة التثبيت الذكية ----------
def pip_install_if_missing(package, import_name=None, install_cmd=None):
    if import_name is None:
        import_name = package.split()[0].split('==')[0].split('>=')[0].split('<')[0]
    try:
        __import__(import_name)
        print(f'✅ {import_name} موجود')
    except ImportError:
        cmd = install_cmd if install_cmd else f'pip install -q {package}'
        !{cmd}
        print(f'⬇️ تم تثبيت {import_name}')

# ---------- المكتبات الأساسية ----------
pip_install_if_missing('pydantic omegaconf imageio pillow tqdm numpy scipy opencv-python PyYAML requests einops')
pip_install_if_missing('transformers')
pip_install_if_missing('diffusers')
pip_install_if_missing('clip', install_cmd='pip install -q git+https://github.com/openai/CLIP.git')
pip_install_if_missing('trimesh')
pip_install_if_missing('lpips')
pip_install_if_missing('nvdiffrast', install_cmd='pip install -q git+https://github.com/NVlabs/nvdiffrast/@335cfa6b33d785730a04283994214bed57884e87 --no-build-isolation')
pip_install_if_missing('moviepy')
pip_install_if_missing('gradio')
pip_install_if_missing('tensorboardX')
pip_install_if_missing('rich')
pip_install_if_missing('jedi')  # لتجنب تحذيرات ipython

# ---------- بناء الامتدادات المحلية (ضمان النجاح) ----------
%cd /content/stable-dreamfusion
extensions = ['raymarching', 'shencoder', 'freqencoder', 'gridencoder']

for ext in extensions:
    if not os.path.isdir(ext):
        print(f'❌ المجلد {ext} غير موجود – فشل الاستنساخ؟')
        continue

    print(f'⏳ بناء {ext} ...')
    # force reinstall للتأكد من استخدام الـ toolkit الصحيح
    build_cmd = f'pip install ./{ext} --no-build-isolation --force-reinstall -q'
    result = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'✅ {ext} تم بناؤه بنجاح')
    else:
        print(f'❌ فشل بناء {ext}:')
        # عرض آخر 15 سطر من الخطأ
        error_lines = result.stderr.strip().split('\n')[-15:]
        print('\n'.join(error_lines))

# ---------- التحقق النهائي: استيراد الامتدادات ----------
print('\n🔍 التحقق من استيراد الامتدادات...')
for ext in extensions:
    try:
        __import__(ext)
        print(f'✅ {ext} قابل للاستيراد')
    except Exception as e:
        print(f'❌ لا يمكن استيراد {ext}: {e}')
        # إذا فشل الامتداد الأساسي، نوقف التنفيذ
        if ext == 'raymarching':
            raise SystemExit('فشل حاسم – لا يمكن المتابعة')

print('🎉 البيئة جاهزة بالكامل وجميع الامتدادات تعمل!')

## 3. واجهة Gradio التفاعلية 🎛️

In [ ]:
import gradio as gr
import subprocess, os, glob

def run_training(prompt, iters, lr, resolution, seed, lambda_entropy, max_steps, workspace):
    cmd = [
        "python", "main.py", "-O",
        "--text", f"'{prompt}'",
        "--workspace", workspace,
        "--iters", str(iters),
        "--lr", str(lr),
        "--w", str(resolution),
        "--h", str(resolution),
        "--seed", str(seed),
        "--lambda_entropy", str(lambda_entropy),
        "--save_mesh",
        "--max_steps", str(max_steps)
    ]
    print(f"🚀 الأمر: {' '.join(cmd)}")
    proc = subprocess.Popen(cmd, cwd="/content/stable-dreamfusion", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    output_lines = []
    for line in proc.stdout:
        output_lines.append(line)
    proc.wait()
    if proc.returncode != 0:
        return f"❌ فشل التدريب. آخر الإخراج:\n{''.join(output_lines[-10:])}", None
    video_pattern = os.path.join(workspace, "results", "*_rgb.mp4")
    videos = sorted(glob.glob(video_pattern), key=os.path.getmtime)
    if videos:
        return f"✅ تم بنجاح! الفيديو: {videos[-1]}", videos[-1]
    else:
        return "✅ انتهى التدريب ولكن لم يتم العثور على فيديو (ربما لم يكتمل التصيير)", None

iface = gr.Interface(
    fn=run_training,
    inputs=[
        gr.Textbox(label="النص الوصفي", value="a DSLR photo of a delicious banana", lines=2),
        gr.Slider(500, 10000, step=100, value=5000, label="التكرارات"),
        gr.Number(value=1e-3, label="معدل التعلم", precision=5),
        gr.Slider(32, 128, step=32, value=64, label="دقة الشبكة"),
        gr.Number(value=12, label="البذرة العشوائية (Seed)"),
        gr.Number(value=1e-4, label="lambda_entropy", precision=5),
        gr.Number(value=512, label="max_steps"),
        gr.Textbox(value="trial", label="اسم مساحة العمل")
    ],
    outputs=[
        gr.Textbox(label="الحالة"),
        gr.Video(label="النموذج الناتج")
    ],
    title="Stable‑Dreamfusion Text‑to‑3D",
    description="حوّل أي نص إلى نموذج ثلاثي الأبعاد. أدخل الإعدادات ثم اضغط **Submit**."
)

iface.launch(debug=True, share=True)

## 4. (اختياري) التشغيل اليدوي

In [ ]:
#@title إعدادات التدريب اليدوي
Prompt_text = "a DSLR photo of a delicious banana"  #@param {type:"string"}
Training_iters = 5000  #@param {type:"integer"}
Learning_rate = 1e-3    #@param {type:"number"}
Training_nerf_resolution = 64  #@param {type:"integer"}
Seed = 12               #@param {type:"integer"}
Lambda_entropy = 1e-4   #@param {type:"number"}
Max_steps = 512         #@param {type:"number"}
Workspace = "trial"     #@param {type:"string"}

%cd /content/stable-dreamfusion
!python main.py -O \
  --text "{Prompt_text}" \
  --workspace {Workspace} \
  --iters {Training_iters} \
  --lr {Learning_rate} \
  --w {Training_nerf_resolution} \
  --h {Training_nerf_resolution} \
  --seed {Seed} \
  --lambda_entropy {Lambda_entropy} \
  --save_mesh \
  --max_steps {Max_steps}

In [ ]:
#@title اختبار النموذج
%cd /content/stable-dreamfusion
!python main.py -O --test --workspace {Workspace} --save_mesh

In [ ]:
#@title عرض الفيديو
from IPython.display import Video
import glob, os
video_pattern = os.path.join(Workspace, "results", "*_rgb.mp4")
videos = sorted(glob.glob(video_pattern), key=os.path.getmtime)
if videos:
    display(Video(videos[-1], embed=True))
else:
    print("لم يتم العثور على فيديو.")